# **Timing Solution Class (TSC)**
## Learning Objectives and Prerequisites
This notebook will show you how to access timing solutions for any valid batch using the Timing Solution Class (TSC) object. It will also show you how to use its various functions which facilitate isolating satellite passes and interpolation over long periods.

To clarify the need for these methods, it will be useful to know what timing solutions are and have some familiarity with how satellites are used to determine them. Moreover, it is also useful to know how baseband data is stored, and how spectra for upchannelized data have a longer time interval.

In [2]:
import os
import sys
from os import path
sys.path.insert(0, "/home/thomasb")
import numpy as np 
import importlib
import matplotlib.pyplot as plt 
from albatros_analysis.src.correlations import timing_solution_class as tsc 
importlib.reload(tsc)

<module 'albatros_analysis.src.correlations.timing_solution_class' from '/home/thomasb/albatros_analysis/src/correlations/timing_solution_class.py'>

## 1. The TSC object 

The first thing to do is set up the object using the start time and data directory.

In [5]:
# Path to timing solutions
root_dir = '/scratch/thomasb/timing_solution'

# This function lets you see what batches have timing solutions
tsc.get_all_batches(root_dir)

[(1753132820, 1753200010),
 (1753200150, 1753286410),
 (1762099360, 1762185620),
 (1762037710, 1762099215),
 (1762185855, 1762267835),
 (1762272160, 1762358430)]

In [ ]:
# Give some start time you want to look at, it must lie within a valid batch
start_ts = 1753210000

# Initialize the object
sol = tsc.TimingSolution(start_ts, root_dir)

# The object has lots of useful information as attributes, for example:
print('Batch start time', sol.batch_start_unix)
print('Batch end time', sol.batch_end_unix)
print('UTC map', sol.UTC_offset)

## 2. Extracting Raw Solutions

All time solutions (for all antenna) are saved alongside the central baseband spectra of the visibility to which it corresponds. Therefore, when you load the raw timing solutions, you must convert from spectrum number to actual UTC time. Fortunately, we have the spectrum to UTC mapping within the TSC object.

In [14]:
# Let's take the 10th data point.
print(f'We have a total of {sol.spectra.shape} time points with data')
spectrum_10 = sol.spectra[10]
print('The 10th data point has spectrum number:', spectrum_10)

# We can manually get the UTC for the 10th data point
UTC_10 = sol.UTC_offset + sol.UTC_per_spec * spectrum_10
print(f'In UTC this corresponds to about {UTC_10:.2f}')

We have a total of (1560,) time points with data
The 10th data point has spectrum number: 278981028.0
In UTC this corresponds to about 1753204699.25


In [15]:
# There is a built in function that does this for you, for any number of spectra you may want
UTC_10 = sol.spectra_to_unix(spectrum_10)
UTC_10 = sol.spectra_to_unix(sol.spectra)

## 3. Interpolation Functions

In [ ]:
#however, most of the time you will have particular time points at which you want a solution. 
#in that case, there are specific built in interpolation functions

#option 1
_, _ = sol.interpolate_delay()

#option 2 
_, _, = sol.interpolate_delay2

In [ ]:
#as a worked example, say I want the timing solution for all antenna for
times_query = np.arange(start_ts, start_ts+1000, 0.1)

#then I can do this
# if lies outside all the pulses, can trigger the extrapolation
# if lies outside the batch (continuous area of data) will also trigger an error
taus_interp = sol.interpolate_delay2(times_query, XXXX, extrapolate=False, break_batch=False)

#this only returns each antenna that is not reference
print(taus_interp.shape)
#so if I want all antenna, correctly computed, then I run:
taus_interp_all = sol.all_blines(taus_interp)
#now I have all antenna (BEWARE THE CONVENTION OF SUBTRACTION)
print(taus_interp_all.shape)

In [ ]:
plt.plot(times_query, taus_interp_all[0, :])

In [ ]:
# If I don't have a specific time array in mind, and instead just want all the regions where there are actual solutions computed, I can use this:
times_existing, taus_existing = sol.interpolate_existing_delays(start_ts, start_ts+10000, 0.1, border_window=50)
#border window tells you how much you can go over existing data (extend the satellite pass data)
#will interpolate the data at any given dt

In [ ]:
plt.scatter(times_existing, taus_existing)